# 22 api coverage

Question: does the tree hold the reference of the plugin API as `okf/api-coverage.md` says: is the
jar that `tests/fetch.py` pins the one of the page, is the committed inventory byte for byte what
`golem-devtool api inventory` writes from that jar, and does the inventory give every number that
the pages name for the reference.

The reference is the jar of spigot-api 1.20.1-R0.1-20230921.163938-66. The notebook holds that it
and its sources jar stand in the cache of the harness with their pins, where `tests/fetch.py` puts
them and git ignores them, writes the inventory again with the `javap` of the pinned JDK and
compares it with the file under `api/`. It counts the inventory with `golem-devtool api numbers`,
counts it a second time in Python, and holds the sentences of the wiki that give a count of the
reference to both. It holds the names of the parameters, which the tool reads from the sources jar,
to the declarations of that jar. It boots no server and runs no class of the jar: `javap` reads the
class files, and the tool reads the two archives itself.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."
# The downloads of tests/fetch.py, a path relative to the root.
cache = "tests/.cache"
# The JDK whose javap prints the classes of the reference. tests/fetch.py installs it here.
java_home = "tests/.cache/jdk-17.0.21+9"

# The pins: the reference, the SHA-256 of its jar and of its sources jar, and the JDK.
reference = "spigot-api-1.20.1-R0.1-20230921.163938-66"
jar_sha256 = "d8aa356781e32896e7d78490813975cac7ca8d99126adc41c2836c56c68b9c3e"
sources_sha256 = "292793da18cba4214c2254748c88ee7463ecf9aa7450a8ba64e39c9d4a6610bf"
java_version = "17.0.21"

## Setup

The cell finds the repository and the `javap` of the pinned JDK, builds `golem-devtool` and prints
the versions of Python, cargo, rustc and javap. The helpers run a subcommand, hash a file and read
a page with every run of white space joined into one blank, so that a search of the page does not
hang on where a line wraps.

In [ ]:
import hashlib
import importlib.util
import json
import re
import shutil
import struct
import subprocess
import sys
import tempfile
import zipfile
import zlib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-22-"))
CACHE = P.root / cache
JAVAP = P.root / java_home / "bin" / "javap"
JAR = CACHE / "spigot-api" / (reference + ".jar")
SOURCES = CACHE / "spigot-api" / (reference + "-sources.jar")
COMMITTED = P.root / "api" / (reference + ".json")
assert JAVAP.is_file(), "%s is missing, python3 -I tests/fetch.py --only jdk installs it" % P.rel(JAVAP)

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"


def devtool(*arguments, expect=0):
    result = run([DEVTOOL, *arguments], cwd=P.root, timeout=300)
    assert result.returncode == expect, "golem-devtool %s exited %d\n--- stdout\n%s--- stderr\n%s" % (
        " ".join(map(str, arguments)), result.returncode, result.stdout[-2000:], result.stderr)
    return (json.loads(result.stdout) if result.stdout.strip() else None), result.stderr.strip()


def page(name):
    return " ".join((P.root / name).read_text(encoding="utf-8").split())


def digest(path, algorithm="sha256"):
    return hashlib.new(algorithm, path.read_bytes()).hexdigest()


def grouped(number):
    return format(number, ",")


PAGES = {name: page(name) for name in ("okf/api-coverage.md", "okf/api.md", "okf/features.md", "okf/decisions.md",
                                       "okf/bukkit-layer.md", "okf/plan.md")}

version = run([JAVAP, "-version"], cwd=P.root, check=True).stdout.strip()
assert version == java_version, "javap is %s, the pin is %s" % (version, java_version)
print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())
print("javap:", version)

## The jar and its sources jar

`python3 -I tests/fetch.py --only spigot-api` downloads both jars from the snapshots repository of
SpigotMC and checks their SHA-256. The notebook downloads nothing: the cell holds that both files
stand in the cache and says what to run where one is missing. It hashes both files itself and holds
the pins of the script and of this notebook to the table of `okf/api-coverage.md` section 1, whose
SHA-1 column it holds as well, and to the hash that `okf/api.md` section 14.1 names. Neither jar is
a file of the tree: the cache is ignored, and git tracks no jar and no file of the cache.

In [ ]:
for path in (JAR, SOURCES):
    assert path.is_file(), "%s is missing, python3 -I tests/fetch.py --only spigot-api fetches it" % P.rel(path)

spec = importlib.util.spec_from_file_location("fetch", P.root / "tests" / "fetch.py")
fetch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(fetch)
assert fetch.SPIGOT_API_JARS == {JAR.name: jar_sha256, SOURCES.name: sources_sha256}, fetch.SPIGOT_API_JARS
assert fetch.SPIGOT_API_VERSION == reference.removeprefix("spigot-api-"), fetch.SPIGOT_API_VERSION

for path, pin in ((JAR, jar_sha256), (SOURCES, sources_sha256)):
    found = digest(path)
    assert found == pin, "%s has the SHA-256 %s, the pin is %s" % (path.name, found, pin)
    row = "| `%s` | `%s` | `%s` |" % (path.name, pin, digest(path, "sha1"))
    assert row in PAGES["okf/api-coverage.md"], "okf/api-coverage.md has no row %s" % row
    print("%s: %d bytes, SHA-256 %s" % (path.name, path.stat().st_size, found))
assert "pins by its SHA-256 `%s`" % jar_sha256 in PAGES["okf/api.md"], "okf/api.md does not name the hash of the jar"

assert "tests/.cache/" in (P.root / ".gitignore").read_text(encoding="utf-8").splitlines(), \
    ".gitignore does not name the cache of the downloads"
if (P.root / ".git").exists():
    tracked = run(["git", "ls-files", "-z"], cwd=P.root, check=True).stdout.split("\0")
    carried = [name for name in tracked if name.endswith(".jar") or name.startswith("tests/.cache/")]
    assert not carried, "git tracks a jar or a file of the cache: %s" % carried
    print("git tracks %d files, no jar and no file of the cache among them" % len([name for name in tracked if name]))

## The inventory

`golem-devtool api inventory` reads every class of the jar with `javap -public`, takes the names
of the parameters from the sources jar and writes the inventory. The cell writes it into a
temporary directory and fails when it differs from the committed file by one byte, with the first
line that differs. The first line of the file names the command that wrote it, its head names both
jars with their hashes, and its types stand in the order of their names.

The file lists the signatures of a jar of the Bukkit project, so `okf/api.md` section 15 gives it
the license of that project and not the one of Golem. The cell reads the tag of the `.license`
file beside it and holds it to a text under `LICENSES/` and to the section of NOTICE that names
the file.

In [ ]:
again = SCRATCH / COMMITTED.name
WRITTEN, _ = devtool("api", "inventory", "--jar", JAR, "--sources", SOURCES, "--javap", JAVAP, "--out", again)
fresh, committed = again.read_bytes(), COMMITTED.read_bytes()
if fresh != committed:
    pairs = enumerate(zip(fresh.split(b"\n"), committed.split(b"\n")), 1)
    at, (new, old) = next(((at, pair) for at, pair in pairs if pair[0] != pair[1]), (0, (b"", b"")))
    raise AssertionError("%s is not what api inventory writes, first in line %d of %d and %d bytes:\n"
                         "  written:   %s\n  committed: %s" % (P.rel(COMMITTED), at, len(fresh), len(committed),
                                                               new.decode()[:300], old.decode()[:300]))
assert WRITTEN["bytes"] == len(committed), WRITTEN["bytes"]

head = committed.split(b"\n", 1)[0].decode()
assert head == '{ "generated-by": "golem-devtool api inventory",', "the first line names no command: %s" % head
INVENTORY = json.loads(committed)
TYPES = INVENTORY["types"]
assert list(INVENTORY) == ["generated-by", "jar", "jar-sha256", "sources", "sources-sha256", "types"], list(INVENTORY)
assert (INVENTORY["jar"], INVENTORY["jar-sha256"]) == (JAR.name, jar_sha256), INVENTORY["jar"]
assert (INVENTORY["sources"], INVENTORY["sources-sha256"]) == (SOURCES.name, sources_sha256), INVENTORY["sources"]
names = [kind["name"] for kind in TYPES]
assert names == sorted(set(names)), "the types do not stand once each in the order of their names"
for kind in TYPES:
    assert list(kind) == ["name", "line", "kind", "modifiers", "extends", "implements", "fields", "methods"], kind["name"]
    assert kind["kind"] in ("class", "interface") and " %s %s" % (kind["kind"], kind["name"]) in " " + kind["line"], \
        kind["line"]

tag = "SPDX-License-" + "Identifier"
beside = COMMITTED.with_name(COMMITTED.name + ".license").read_text(encoding="utf-8")
licenses = re.findall(r"^%s: (.+)$" % tag, beside, flags=re.M)
assert len(licenses) == 1 and licenses[0] != "AGPL-3.0-only", "the inventory takes the license of Golem: %s" % licenses
assert (P.root / "LICENSES" / (licenses[0] + ".txt")).is_file(), "LICENSES/ holds no text of %s" % licenses[0]
notice = (P.root / "NOTICE").read_text(encoding="utf-8").split("\n\n\n")
section = [part for part in notice if "\nFiles: %s\n" % P.rel(COMMITTED) in part]
assert len(section) == 1 and "\nLicense: %s\n" % licenses[0] in section[0], "NOTICE does not name the inventory"
print("%s: %d bytes, byte for byte what api inventory writes, under %s" % (P.rel(COMMITTED), len(committed), licenses[0]))

## The numbers of the pages

`golem-devtool api numbers` counts a written inventory, and `api inventory` prints the same counts
of what it wrote, with the classes of the jar and those it left out, one `package-info` for each
package. The cell counts the inventory a second time in Python, from the JSON alone: the types, the
methods and the fields, the types that are public and those that are anonymous, the packages, the
interfaces of the entity tree with its leaves, and the event classes with those that are concrete,
that a listener can cancel and that a listener can name. No type holds a line twice, which is what
lets a mapping file key its members by their lines.

Then it holds the pages to the counts. Every sentence of the wiki that gives a number of the
reference is written here with the number taken from the tool, and the cell fails when a page does
not hold the sentence.

In [ ]:
NUMBERS, _ = devtool("api", "numbers", COMMITTED)
assert {key: WRITTEN[key] for key in NUMBERS} == NUMBERS, "api inventory and api numbers count differently"
assert (WRITTEN["classesOfJar"], WRITTEN["leftOut"]) == (NUMBERS["types"] + NUMBERS["packages"], NUMBERS["packages"]), \
    "the inventory leaves out more or less than one package-info for each package"

EVENT, ENTITY, CANCELLABLE = "org.bukkit.event.Event", "org.bukkit.entity.Entity", "org.bukkit.event.Cancellable"
HANDLER_LIST = "public static org.bukkit.event.HandlerList getHandlerList()"
BY_NAME = {kind["name"]: kind for kind in TYPES}


def home(kind):
    return kind["name"].rpartition(".")[0]


def ancestors(name):
    found, waiting = set(), [name]
    while waiting:
        kind = BY_NAME.get(waiting.pop())
        for parent in (kind["extends"] + kind["implements"]) if kind else ():
            parent = parent.split("<")[0]
            if parent not in found:
                found.add(parent)
                waiting.append(parent)
    return found


def chain(kind):
    while kind is not None:
        if any(method["line"] == HANDLER_LIST for method in kind["methods"]):
            return kind["name"]
        kind = BY_NAME.get(kind["extends"][0].split("<")[0]) if kind["extends"] else None
    return None


def method_name(line):
    return line[:line.index("(")].split(" ")[-1]


for kind in TYPES:
    lines = [method["line"] for method in kind["methods"]] + kind["fields"]
    assert len(set(lines)) == len(lines), "%s holds a line twice" % kind["name"]
    assert all(line.startswith("public ") and not line.endswith(";") for line in lines), kind["name"]
    assert all("(" in method["line"] for method in kind["methods"]), kind["name"]
    assert not any("(" in field for field in kind["fields"]), kind["name"]

methods = [method for kind in TYPES for method in kind["methods"]]
fields = [field for kind in TYPES for field in kind["fields"]]
anonymous = [kind for kind in TYPES if re.search(r"\$\d+$", kind["name"])]
shared = {}
for kind in TYPES:
    shared.setdefault(kind["name"].rpartition(".")[2], set()).add(home(kind))
of_entities = [kind for kind in TYPES if home(kind) == "org.bukkit.entity" or home(kind).startswith("org.bukkit.entity.")]
interfaces = [kind for kind in of_entities if kind["kind"] == "interface"]
tree = {kind["name"] for kind in interfaces if kind["name"] == ENTITY or ENTITY in ancestors(kind["name"])}
above = {name: ancestors(name) for name in tree}
leaves = [name for name in tree if not any(name in above[other] for other in tree)]
events = [kind for kind in TYPES if kind["name"] == EVENT or EVENT in ancestors(kind["name"])]
concrete = [kind for kind in events if "abstract" not in kind["modifiers"]]
listenable = [kind for kind in events if chain(kind)]
setters = [method_name(method["line"]) for kind in events for method in kind["methods"]]
recount = {
    "types": len(TYPES), "methods": len(methods), "fields": len(fields),
    "members": len(TYPES) + len(methods) + len(fields),
    "classes": sum(kind["kind"] == "class" for kind in TYPES),
    "interfaces": sum(kind["kind"] == "interface" for kind in TYPES),
    "public": sum("public" in kind["modifiers"] for kind in TYPES),
    "anonymous": {"types": len(anonymous), "methods": sum(len(kind["methods"]) for kind in anonymous)},
    "packages": len({home(kind) for kind in TYPES}),
    "orgBukkit": sum(home(kind) == "org.bukkit" for kind in TYPES),
    "sharedNames": sum(len(homes) > 1 for homes in shared.values()),
    "entity": {
        "types": len(of_entities), "methods": sum(len(kind["methods"]) for kind in of_entities),
        "interfaces": len(interfaces), "interfaceMethods": sum(len(kind["methods"]) for kind in interfaces),
        "tree": len(tree), "leaves": len(leaves),
    },
    "events": {
        "classes": len(events), "concrete": len(concrete),
        "concreteCancellable": sum(CANCELLABLE in ancestors(kind["name"]) for kind in concrete),
        "listenable": len(listenable),
        "ownHandlerList": sum(chain(kind) == kind["name"] for kind in events),
        "setters": sum(bool(re.match(r"set[A-Z]", name)) and name != "setCancelled" for name in setters),
    },
}
for key, counted in recount.items():
    assert NUMBERS[key] == counted, "api numbers counts %s as %s, the cell as %s" % (key, NUMBERS[key], counted)
abstract_listenable = sorted(kind["name"].rpartition(".")[2] for kind in listenable if kind not in concrete)
assert abstract_listenable == ["InventoryInteractEvent", "PlayerChannelEvent"], abstract_listenable
living = BY_NAME["org.bukkit.entity.LivingEntity"]

N, E, V = NUMBERS, NUMBERS["entity"], NUMBERS["events"]
surface = (grouped(N["types"]), grouped(N["methods"]), grouped(N["fields"]))
SAID = [
    ("okf/api-coverage.md", "but the `package-info` of its %d packages" % N["packages"]),
    ("okf/api-coverage.md", "%s types, %s methods and %s fields and constants" % surface),
    ("okf/api-coverage.md", "%s of the types are public, %d are anonymous classes and %d are classes with a name "
                            "that are not public" % (grouped(N["public"]), N["anonymous"]["types"],
                                                     N["types"] - N["public"] - N["anonymous"]["types"])),
    ("okf/api-coverage.md", "of each of its members, %s files" % grouped(N["types"])),
    ("okf/api-coverage.md", "The %d types of the package `org.bukkit` itself" % N["orgBukkit"]),
    ("okf/api-coverage.md", "The %d packages of the inventory" % N["packages"]),
    ("okf/api-coverage.md", "one of the %d anonymous classes of the jar, with their %d methods"
                            % (N["anonymous"]["types"], N["anonymous"]["methods"])),
    ("okf/api-coverage.md", "that a listener can name, %d of the %d" % (V["listenable"], V["classes"])),
    ("okf/api-coverage.md", "The overview of the %d listenable events" % V["listenable"]),
    ("okf/api-coverage.md", "with five of its %d members" % (len(living["methods"]) + len(living["fields"]))),
    ("okf/api.md", "%s types, %s public methods, %s fields and constants and %d events" % (*surface, V["classes"])),
    ("okf/api.md", "over all %s members of the inventory: %s types, %s methods and %s fields and constants"
                   % (grouped(N["members"]), *surface)),
    ("okf/api.md", "holds %d interfaces in `org.bukkit.entity` and its subpackages, %d in the tree of `Entity` with "
                   "`Entity` itself and %d of them leaves" % (E["interfaces"], E["tree"], E["leaves"])),
    ("okf/api.md", "The %d interfaces of `org.bukkit.entity` and its subpackages are GraphQL interfaces, and the "
                   "object types are the %d leaves" % (E["interfaces"], E["leaves"])),
    ("okf/api.md", "The %d interfaces and %d classes of `org.bukkit.entity` and its subpackages with their %s methods"
                   % (E["interfaces"], E["types"] - E["interfaces"], grouped(E["methods"]))),
    ("okf/api.md", "holds %d simple names that more than one package uses" % N["sharedNames"]),
    ("okf/api.md", "holds %d event classes: %d concrete, %d of them cancellable, and %d abstract"
                   % (V["classes"], V["concrete"], V["concreteCancellable"], V["classes"] - V["concrete"])),
    ("okf/api.md", "its setters, %d besides `setCancelled` over all events" % V["setters"]),
    ("okf/api.md", "may name %d classes: the %d concrete ones and the abstract `InventoryInteractEvent` and "
                   "`PlayerChannelEvent`" % (V["listenable"], V["concrete"])),
    ("okf/api.md", "%d classes declare their own handler list" % V["ownHandlerList"]),
    ("okf/api.md", "the %d methods of the %d anonymous classes" % (N["anonymous"]["methods"], N["anonymous"]["types"])),
    ("okf/api.md", "the overview of the %d listenable events" % V["listenable"]),
    ("okf/features.md", "the %d interfaces and %d classes of the entity packages with %s methods"
                        % (E["interfaces"], E["types"] - E["interfaces"], grouped(E["methods"]))),
    ("okf/decisions.md", "%s types, %s public methods, %s fields and constants and %d events" % (*surface, V["classes"])),
    ("okf/bukkit-layer.md", "%s types, %s public methods, %s fields and constants and %d event classes, %d of them "
                            "concrete" % (*surface, V["classes"], V["concrete"])),
    ("okf/plan.md", "%s types, %s public methods, %s fields and constants and %d events" % (*surface, V["classes"])),
]
for name, sentence in SAID:
    assert sentence in PAGES[name], "%s does not say: %s" % (name, sentence)
print(json.dumps(NUMBERS, indent=1))
print("the cell counts the same, and %d sentences of %d pages give these numbers" % (len(SAID), len({name for name, _ in SAID})))

## The names of the parameters

javap prints the types of the parameters of a method and not their names, so the tool reads the
names from the declarations of the sources jar: a method without parameters has none, a method
with parameters has one name for each, and `names` is `null` where the sources declare no such
method. The cell holds a few methods to their names and reads the declaration of each from the
sources jar itself: a setter, a method of three numbers, a constructor, a generic method, a method
with a variable count of arguments, two overloads that differ in one type, a method of an inner
class, and a bridge method, which the compiler adds for a method with a type parameter and which
takes the names of that method.

The methods without names are the ones `okf/api-coverage.md` section 1.1 lists: the
`valueOf(String)` that the compiler adds to every enum, where `MapView$Scale` also declares a
`valueOf(byte)` of its own that must not lend it a name, the methods of the anonymous classes, and
two methods that `YamlRepresenter` does not declare itself.

In [ ]:
NAMED = [
    ("org.bukkit.entity.Damageable", "public abstract void setHealth(double)", ["health"],
     r"void setHealth\(double health\);"),
    ("org.bukkit.World", "public abstract org.bukkit.block.Block getBlockAt(int, int, int)", ["x", "y", "z"],
     r"Block getBlockAt\(int x, int y, int z\);"),
    ("org.bukkit.Location",
     "public org.bukkit.Location(org.bukkit.World, double, double, double, float, float)",
     ["world", "x", "y", "z", "yaw", "pitch"],
     r"public Location\((@\w+ )?(final )?World world, (final )?double x, (final )?double y, (final )?double z, "
     r"(final )?float yaw, (final )?float pitch\)"),
    ("org.bukkit.RegionAccessor",
     "public abstract <T extends org.bukkit.entity.Entity> T spawn(org.bukkit.Location, java.lang.Class<T>) "
     "throws java.lang.IllegalArgumentException", ["location", "clazz"],
     r"<T extends Entity> T spawn\(@NotNull Location location, @NotNull Class<T> clazz\) throws"),
    ("org.bukkit.command.CommandSender", "public abstract void sendMessage(java.lang.String...)", ["messages"],
     r"void sendMessage\(@NotNull String\.\.\. messages\);"),
    ("org.bukkit.Bukkit",
     "public static org.bukkit.util.CachedServerIcon loadServerIcon(java.io.File) "
     "throws java.lang.IllegalArgumentException, java.lang.Exception", ["file"],
     r"CachedServerIcon loadServerIcon\(@NotNull File file\) throws"),
    ("org.bukkit.Bukkit",
     "public static org.bukkit.util.CachedServerIcon loadServerIcon(java.awt.image.BufferedImage) "
     "throws java.lang.IllegalArgumentException, java.lang.Exception", ["image"],
     r"CachedServerIcon loadServerIcon\(@NotNull BufferedImage image\) throws"),
    ("org.bukkit.entity.Player$Spigot",
     "public void sendMessage(net.md_5.bungee.api.ChatMessageType, java.util.UUID, "
     "net.md_5.bungee.api.chat.BaseComponent...)", ["position", "sender", "components"],
     r"void sendMessage\(@NotNull net\.md_5\.bungee\.api\.ChatMessageType position, @Nullable java\.util\.UUID "
     r"sender, @NotNull net\.md_5\.bungee\.api\.chat\.BaseComponent\.\.\. components\)"),
    ("org.bukkit.plugin.RegisteredServiceProvider", "public int compareTo(java.lang.Object)", ["other"],
     r"public int compareTo\(@NotNull RegisteredServiceProvider<\?> other\)"),
    ("org.bukkit.map.MapView$Scale", "public static org.bukkit.map.MapView$Scale valueOf(byte)", ["value"],
     r"public static Scale valueOf\(byte value\)"),
    ("org.bukkit.map.MapView$Scale", "public static org.bukkit.map.MapView$Scale valueOf(java.lang.String)", None, None),
    ("org.bukkit.GameMode", "public static org.bukkit.GameMode valueOf(java.lang.String)", None, None),
]
with zipfile.ZipFile(SOURCES) as archive:
    for kind, line, expected, declared in NAMED:
        found = [method["names"] for method in BY_NAME[kind]["methods"] if method["line"] == line]
        assert found == [expected], "%s, %s: the inventory names %s, the cell %s" % (kind, line, found, expected)
        if declared is not None:
            outer = kind.split("$")[0].replace(".", "/") + ".java"
            source = " ".join(archive.read(outer).decode("utf-8").split())
            assert re.search(declared, source), "%s does not declare %s" % (outer, declared)

identifier = re.compile(r"^[A-Za-z_$][A-Za-z0-9_$]*$")
asked = unnamed = 0
for kind in TYPES:
    for method in kind["methods"]:
        inside = method["line"][method["line"].index("(") + 1:method["line"].index(")")]
        depth, count = 0, 1 if inside else 0
        for character in inside:
            depth += {"<": 1, ">": -1}.get(character, 0)
            count += character == "," and depth == 0
        if count == 0:
            assert method["names"] == [], "%s, %s: names for no parameter" % (kind["name"], method["line"])
            continue
        asked += 1
        if method["names"] is None:
            unnamed += 1
            continue
        assert len(method["names"]) == count and all(identifier.match(name) for name in method["names"]), \
            "%s, %s: %s" % (kind["name"], method["line"], method["names"])
assert NUMBERS["names"] == {"asked": asked, "named": asked - unnamed}, (NUMBERS["names"], asked, unnamed)

nameless = {(kind["name"], method["line"]) for kind in TYPES for method in kind["methods"] if method["names"] is None}
enums = [kind["name"] for kind in TYPES if any(parent.startswith("java.lang.Enum<") for parent in kind["extends"])]
value_of = {(name, "public static %s valueOf(java.lang.String)" % name) for name in enums}
assert value_of <= nameless, "an enum whose valueOf(String) carries names: %s" % sorted(value_of - nameless)[:3]
of_anonymous = {pair for pair in nameless - value_of if re.search(r"\$\d+$", pair[0])}
others = sorted(nameless - value_of - of_anonymous)
assert [kind for kind, _ in others] == ["org.bukkit.configuration.file.YamlRepresenter"] * 2, others
sentence = ("which is so for %d of the %s methods with parameters: the `valueOf(String)` that the compiler adds to "
            "each of the %d enums, %d methods of anonymous classes and %d methods that `YamlRepresenter` does not "
            "declare itself" % (unnamed, grouped(asked), len(value_of), len(of_anonymous), len(others)))
assert sentence in PAGES["okf/api-coverage.md"], "okf/api-coverage.md does not say: %s" % sentence
print("names: %d of %d methods with parameters carry them, %d do not: %d valueOf of an enum, %d of anonymous classes, %d others"
      % (asked - unnamed, asked, unnamed, len(value_of), len(of_anonymous), len(others)))

## What the tool refuses

The tool writes no inventory from what it cannot read whole. The cell gives it a call without a
flag, a jar that is cut short, a sources jar with one byte changed inside the packed bytes of one
file, whose packing then fails, the same jar with the CRC-32 of that file changed in its directory,
a `javap` that prints a member that is not public, and one that ends with an error, and it holds
that no file is written in any of these cases. `api numbers` refuses a file that another command
wrote, a file with a key it does not know, and a file in which two classes extend each other, which
no jar holds and which would send a walk up the classes round in a circle.

A directory may name any size for a file. The cell builds two small archives of its own, a jar with
one class and a sources jar with its one Java file, and the tool reads them and names the parameter
of the constructor. Then the directory of the sources jar claims 4 GiB for that file, and the tool
runs under a limit of 1 GiB of address space where the system sets one: it ends with its own
message, since it takes no more memory on the word of a directory than the packed bytes can fill.

In [ ]:
out = SCRATCH / "refused.json"


def refused(*arguments):
    _, error = devtool("api", *arguments, expect=1)
    assert not out.exists(), "a refused call wrote %s" % out.name
    return error


error = refused("inventory", "--jar", JAR, "--sources", SOURCES, "--javap", JAVAP)
assert error == ("golem-devtool: api inventory: api inventory takes --jar <path> --sources <path> --javap <path> "
                 "--out <path>"), error
error = refused("inventory", "--jar", JAR, "--jar", JAR, "--sources", SOURCES, "--javap", JAVAP, "--out", out)
assert error.endswith("--out <path>"), error

short = SCRATCH / "short.jar"
short.write_bytes(JAR.read_bytes()[:4000])
error = refused("inventory", "--jar", short, "--sources", SOURCES, "--javap", JAVAP, "--out", out)
assert error.endswith("short.jar: no end record of a zip archive"), error

victim = "org/bukkit/entity/Damageable.java"
packed = bytearray(SOURCES.read_bytes())
with zipfile.ZipFile(SOURCES) as archive:
    entry = archive.getinfo(victim)
name_length, extra_length = struct.unpack("<HH", packed[entry.header_offset + 26:entry.header_offset + 30])
assert entry.compress_type == zipfile.ZIP_DEFLATED and entry.compress_size > 64, entry
packed[entry.header_offset + 30 + name_length + extra_length + 32] ^= 0x10
broken = SCRATCH / "broken-sources.jar"
broken.write_bytes(packed)
error = refused("inventory", "--jar", JAR, "--sources", broken, "--javap", JAVAP, "--out", out)
assert error.startswith("golem-devtool: api inventory: %s: " % victim), error
print("a changed byte in the sources jar:", error.split(": ", 2)[2])


def fake_javap(name, script):
    path = SCRATCH / name
    path.write_text("#!/bin/sh\n" + script, encoding="utf-8")
    path.chmod(0o755)
    return path


hidden = fake_javap("javap-hidden", "cat <<'END'\nCompiled from \"A.java\"\npublic class a.A {\n  int hidden;\n}\nEND\n")
error = refused("inventory", "--jar", JAR, "--sources", SOURCES, "--javap", hidden, "--out", out)
assert error == "golem-devtool: api inventory: javap: no public member in the line:   int hidden;", error
failing = fake_javap("javap-failing", "echo 'Error: class not found: a.A' >&2\nexit 1\n")
error = refused("inventory", "--jar", JAR, "--sources", SOURCES, "--javap", failing, "--out", out)
assert "javap ended with" in error and error.endswith("wrote: Error: class not found: a.A"), error
one = fake_javap("javap-one", "cat <<'END'\nCompiled from \"A.java\"\npublic class a.A {\n  public a.A();\n}\nEND\n")
error = refused("inventory", "--jar", JAR, "--sources", SOURCES, "--javap", one, "--out", out)
assert error == "golem-devtool: api inventory: javap printed 1 classes of the %d asked" % NUMBERS["types"], error

text = COMMITTED.read_text(encoding="utf-8")
by_hand = SCRATCH / "by-hand.json"
by_hand.write_text(text.replace('"generated-by": "golem-devtool api inventory"', '"generated-by": "hand"', 1),
                   encoding="utf-8")
error = refused("numbers", by_hand)
assert error.endswith("by-hand.json: written by hand, not by golem-devtool api inventory"), error
unknown = SCRATCH / "unknown.json"
unknown.write_text(text.replace('"jar":', '"note": "", "jar":', 1), encoding="utf-8")
error = refused("numbers", unknown)
assert "unknown.json: unknown field `note`" in error, error
error = refused("numbers")
assert error == "golem-devtool: api numbers: api numbers takes one inventory", error

directory = bytearray(SOURCES.read_bytes())
# The record at the end of an archive names the place of its directory.
at = struct.unpack("<I", directory[directory.rindex(b"PK\x05\x06") + 16:][:4])[0]
while True:
    assert directory[at:at + 4] == b"PK\x01\x02", "the directory of the sources jar ends before %s" % victim
    name_length, extra_length, comment_length = struct.unpack("<HHH", directory[at + 28:at + 34])
    if directory[at + 46:at + 46 + name_length].decode("utf-8") == victim:
        break
    at += 46 + name_length + extra_length + comment_length
for place in range(at + 16, at + 20):
    directory[place] ^= 0xFF
miscounted = SCRATCH / "miscounted-sources.jar"
miscounted.write_bytes(directory)
error = refused("inventory", "--jar", JAR, "--sources", miscounted, "--javap", JAVAP, "--out", out)
assert error == "golem-devtool: api inventory: %s: its bytes miss the size or the CRC-32 of the directory" % victim, error


def small_archive(name, content, *, claimed=None):
    # A zip archive of one deflated file, whose directory names `claimed` bytes for it.
    packer = zlib.compressobj(9, zlib.DEFLATED, -15)
    packed = packer.compress(content) + packer.flush()
    name, size, crc = name.encode("utf-8"), len(content) if claimed is None else claimed, zlib.crc32(content)
    local = struct.pack("<IHHHHHIIIHH", 0x04034B50, 20, 0, 8, 0, 0, crc, len(packed), size, len(name), 0) + name + packed
    central = struct.pack("<IHHHHHHIIIHHHHHII", 0x02014B50, 20, 20, 0, 8, 0, 0, crc, len(packed), size, len(name),
                          0, 0, 0, 0, 0, 0) + name
    return local + central + struct.pack("<IHHHHIIH", 0x06054B50, 0, 0, 1, 1, len(central), len(local), 0)


# 1 GiB of address space for the tool, where the shell can set that limit.
LIMITED = ["sh", "-c", 'ulimit -v 1048576 2>/dev/null; exec "$@"', "sh"]

small_jar, small_sources = SCRATCH / "small.jar", SCRATCH / "small-sources.jar"
small_jar.write_bytes(small_archive("a/A.class", b""))
small_javap = fake_javap("javap-small", "cat <<'END'\nCompiled from \"A.java\"\npublic class a.A {\n  public a.A(int);\n}\nEND\n")
SMALL = [DEVTOOL, "api", "inventory", "--jar", small_jar, "--sources", small_sources, "--javap", small_javap, "--out", out]
small_sources.write_bytes(small_archive("a/A.java", b"package a;\n\npublic class A {\n    public A(int count) {\n    }\n}\n"))
result = subprocess.run(LIMITED + SMALL, capture_output=True, text=True, timeout=300)
assert result.returncode == 0, "the tool does not read the small archives of the cell:\n" + result.stderr
small = json.loads(out.read_text(encoding="utf-8"))["types"]
assert [(kind["name"], kind["methods"]) for kind in small] == \
    [("a.A", [{"line": "public a.A(int)", "names": ["count"]}])], small
out.unlink()
small_sources.write_bytes(small_archive("a/A.java", b"package a;\n", claimed=0xFFFFFFFE))
result = subprocess.run(LIMITED + SMALL, capture_output=True, text=True, timeout=300)
assert result.returncode == 1 and result.stderr.strip() == \
    "golem-devtool: api inventory: a/A.java: its bytes miss the size or the CRC-32 of the directory", \
    "a claim of 4 GiB ends the tool with the status %d and: %s" % (result.returncode, result.stderr.strip()[-300:])
assert not out.exists(), "a claim of 4 GiB wrote an inventory"

looped = SCRATCH / "looped.json"
player = '"name": "org.bukkit.event.player.PlayerEvent",'
block = '"name": "org.bukkit.event.block.BlockEvent",'
parent = '"extends": ["org.bukkit.event.Event"],'
at_player, at_block = text.index(player), text.index(block)
assert at_block < at_player and text.index(parent, at_block) < at_player, "the two classes stand otherwise in the inventory"
turned = text[:at_player] + text[at_player:].replace(parent, '"extends": ["org.bukkit.event.block.BlockEvent"],', 1)
turned = turned[:at_block] + turned[at_block:].replace(parent, '"extends": ["org.bukkit.event.player.PlayerEvent"],', 1)
looped.write_text(turned, encoding="utf-8")
assert turned.count(parent) == text.count(parent) - 2
error = refused("numbers", looped)
assert error.endswith("looped.json: org.bukkit.event.block.BlockEvent stands among the types that it extends or implements"), error
print("refused: a call without a flag, a short jar, a sources jar with a changed byte and one with a changed CRC-32, "
      "a claim of 4 GiB, three javap and three inventories")

## Verdict

The cell clears the temporary directory. A run that reaches it passed every section.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: the pinned jar of spigot-api and its sources jar are the ones of the pages, the committed inventory "
      "is byte for byte what golem-devtool api inventory writes from them, it gives every number that the pages "
      "name for the reference, and its methods carry the names of their parameters as the sources jar declares them")